# 2.1 Differential expression analysis
## 2.1 遺伝子発現変動解析
Updated 2026-06-23  
更新日 2026-06-23  

> この文書の日本語訳は AI（Claude Opus 4.8）によって作成されました。正式な原文は英語版です。誤りや不自然な表現に気づいた場合は、英語版を参照してください。

Data are mouse brain, *Atp8a1* single-knockout (KO) vs *Atp8a1/2* double-knockout (DKO), sampled at two developmental timepoints: P0 (day of birth) and P14 (two weeks later). 5 biological replicates per genotype per timepoint (total 20 samples).  
データはマウスの脳で、*Atp8a1* 単独ノックアウト（KO）と *Atp8a1/2* 二重ノックアウト（DKO）を、2つの発生時点（P0：出生当日、P14：2週間後）で採取したものです。遺伝子型・時点ごとに生物学的反復が5つずつ（合計20サンプル）あります。

For more information, see the **[DESeq2 docs](https://bioconductor.org/packages/devel/bioc/vignettes/DESeq2/inst/doc/DESeq2.html)**.  
詳細は **[DESeq2 ドキュメント](https://bioconductor.org/packages/devel/bioc/vignettes/DESeq2/inst/doc/DESeq2.html)** を参照してください。

This notebook works through the following analyses:  
このノートブックでは、次の解析を順に進めます。

2.1.1. **PCA** of all samples, to look for structure, outliers, and the dominant sources of variation;  
       全サンプルの **PCA**。構造、外れ値、そして分散の主な要因を調べる；  
2.1.2. **Pairwise** KO-vs-DKO comparisons, run separately within each timepoint (P0, then P14);  
       各時点（P0、続いて P14）ごとに別々に行う、KO 対 DKO の **ペアワイズ** 比較；  
2.1.3. A **multifactorial** model using all samples at once, including a test for a genotype × timepoint *interaction*,  asking whether effect of the knockout depends on developmental stage;  
       全サンプルを一度に使う **多因子** モデル。遺伝子型 × 時点の *交互作用* の検定を含み、ノックアウトの効果が発生段階に依存するかを問う；  
2.1.4. Plotting differential genes using a **volcano plot**.  
       **ボルケーノプロット** を使って発現変動遺伝子を可視化する。

## Dependencies
### 依存パッケージ
Dependencies are listed in `differential-expression.yml`. Install with `conda` or using the included `install.sh` script.  
依存パッケージは `differential-expression.yml` に記載されています。`conda` を使うか、付属の `install.sh` スクリプトを使ってインストールしてください。

## Required input files
### 必要な入力ファイル
Put these files in the `data` directory.  
これらのファイルを `data` ディレクトリに置いてください。

**atp8a2.expected_counts.tsv**: expected transcript counts from `RSEM`, gene × sample. Row names are of the form `ENSMUSG00000000001_Gnai3`. Data are not yet published; ask Owen for the download link. **Do not redistribute**.  
**atp8a2.expected_counts.tsv**：`RSEM` による expected transcript counts（遺伝子 × サンプル）。行名は `ENSMUSG00000000001_Gnai3` の形式です。データは未発表です。ダウンロードリンクは Owen に問い合わせてください。**再配布しないでください**。

**sample_metadata.tsv**: sample annotations `treatment` (KO/DKO), `timepoint` (P0/P14). Included in this repo.  
**sample_metadata.tsv**：サンプルの注釈 `treatment`（KO/DKO）、`timepoint`（P0/P14）。このリポジトリに含まれています。

**Mus_musculus.GRCm39.113.gtf**: Ensembl gene annotation (release 113, GRCm39). Download from https://ftp.ensembl.org/pub/release-113/gtf/mus_musculus/Mus_musculus.GRCm39.113.gtf.gz.  
**Mus_musculus.GRCm39.113.gtf**：Ensembl の遺伝子アノテーション（release 113、GRCm39）。https://ftp.ensembl.org/pub/release-113/gtf/mus_musculus/Mus_musculus.GRCm39.113.gtf.gz からダウンロードしてください。

In [ ]:
# Load dependencies

Sys.setenv(LANGUAGE = "en") # set language to "ja" if you prefer
library(DESeq2)             # differential expression
library(plyranges)          # genomic regions
library(tidyverse)          # tables
library(ggplot2)            # plotting
library(ggrepel)            # pretty labels on plots
library(extrafont)          # Use Arial font
library(svglite)            # save .svg files
library(patchwork)          # combine plots

getwd()                     # print the working directory
sessionInfo()               # print all library versions

In [ ]:
# Plotting defaults
suppressMessages(extrafont::font_import(pattern="Arial",prompt=FALSE))
suppressMessages(extrafont::loadfonts())

base_theme <- theme_classic(base_size=14, base_family="Arial",) +
    theme(axis.text = element_text(size=14,colour="black"),
          aspect.ratio=1,
         )
theme_set(base_theme)

write_plot <- function(plt,outfile,width,height){
    b=basename(outfile)
    d=dirname(outfile)
    dir.create(d, recursive=TRUE, showWarnings = FALSE)
    pdf.options(encoding='ISOLatin2.enc')
    pngName = paste(b, ".png", sep="")
    svgName = paste(b, ".svg", sep = "")
    ggsave(path=d, device="png", filename=pngName, width=width, height=height, units='in')
    ggsave(path=d, device="svg", filename=svgName, width=width, height=height, units='in')
}

# 2.1.0 Get and process data
## 2.1.0 データの取得と前処理

In [ ]:
data_location = 'data/atp8a2.expected_counts.tsv'
metadata_location = 'data/sample_metadata.tsv'
annotation_location = 'data/Mus_musculus.GRCm39.113.gtf'

In [ ]:
# Read and preprocess expression data

load_annotations <- function(annotation_path) {
    # Load gene annotations from a .gtf file.
    g <- rtracklayer::import(annotation_path) %>%
        filter(type=='gene') %>%
        filter(gene_biotype=='protein_coding')
    return(g)
}

filter_protein_coding <- function(gene_matrix,annotation_path){
    # given a matrix with rownames of the form ENSMUSG00000000001_Gnai3,
    # return a subset consisting of only rows where the stable gene ID ENSMUSG00000000001
    # is annotated as a protein-coding gene.
    message('filtering for protein-coding genes ...')
    g <- load_annotations(annotation_path)
    # need to match on stable ENSMUSG IDs
    ensembl_ids <- sub("_.*", "", rownames(gene_matrix))
    keep <- ensembl_ids %in% g$gene_id
    filtered_data <- gene_matrix[keep, ]
    return(filtered_data)
}

load_gex  <- function(data_location,annotation_location){
    message("loading gene expression data from ", data_location, "...")
    cts = as.matrix(read.csv(data_location,sep='\t',row.names="gene_id",check.names=FALSE))
    cts = round(cts) # counts must be integer
    cts = filter_protein_coding(cts,annotation_location)
    new_rownames <- sub("^ENSMUSG\\d*_", "", rownames(cts)) # Remove ENSMUSG prefixes, keep symbol
    rownames(cts) <- new_rownames
    return(cts)
}

data = load_gex(data_location,annotation_location)
dim(data) # # of genes and samples
data %>% head # first 5 rows

In [ ]:
# Read and format sample metadata
meta = read.table(metadata_location, row.names=1, header=TRUE)
meta <- meta[match(colnames(data), rownames(meta)), , drop=FALSE]

meta <- meta %>%
    mutate(treatment = relevel(factor(treatment), ref="KO"),
           timepoint = relevel(factor(timepoint), ref="P0")
          )

meta

# 2.1.1 Principal components analysis (PCA) on all samples
## 2.1.1 全サンプルに対する主成分分析（PCA）

## Purpose
### 目的
Do samples cluster by genotype (KO vs DKO) or by timepoint (P0 vs P14) in the first two PCs?  
最初の2つの主成分で、サンプルは遺伝子型（KO 対 DKO）や時点（P0 対 P14）ごとにまとまるか？  
Are any samples outliers?  
外れ値となるサンプルはあるか？  
Which factor explains the most variance?  
最も分散を説明する要因はどれか？

## Results
### 結果
PC1 (93% of variance) separates P0 and P14.  
PC1（分散の93%）が P0 と P14 を分離する。

## Outputs
### 出力
`results/atp8a2/pca_all.{png,svg}`

In [ ]:
dds <- DESeq2::DESeqDataSetFromMatrix(countData = data,
                              colData = meta,
                              design = ~ timepoint + treatment)
# Pre-filter: keep genes with >= 30 counts in at least a full group's worth of samples.
smallestGroupSize <- 5
keep <- rowSums(counts(dds) >= 30) >= smallestGroupSize
dds <- dds[keep,]
# Run regression
dds <- DESeq(dds)
# Print model summary
dds

In [ ]:
# print matrix size after pre-filtering
dim(counts(dds))

In [ ]:
# PCA on all samples, colored by genotype and shaped by timepoint.
vsd <- vst(dds, blind=TRUE)

pcaData <- plotPCA(vsd, intgroup=c("treatment","timepoint"), returnData=TRUE)
pv <- round(100 * attr(pcaData, "percentVar"))

a = ggplot(pcaData, aes(PC1, PC2, color=treatment, shape=timepoint)) +
    geom_point(size = 3) +
    geom_label_repel(aes(label = name), size = 2.5, max.overlaps = Inf) +
    xlab(paste0("PC1: ", pv[1], "% variance")) +
    ylab(paste0("PC2: ", pv[2], "% variance"))
w=8;h=8
write_plot(a,outfile='results/atp8a2/pca_all',width=w,height=h)
options(repr.plot.width=w, repr.plot.height=h)
a

# 2.1.2 Pairwise comparisons within each timepoint
## 2.1.2 各時点内でのペアワイズ比較

The simplest way to ask "what does the DKO do?" is to compare KO vs DKO *within a single timepoint*, one timepoint at a time. Below we run two independent analyses: KO vs DKO at P0, then KO vs DKO at P14.  
「DKO は何をするのか？」を問う最も単純な方法は、*1つの時点の中で* KO と DKO を、時点ごとに1つずつ比較することです。以下では、2つの独立した解析を実行します。P0 での KO 対 DKO、続いて P14 での KO 対 DKO です。

Because each analysis uses samples from only one timepoint, we only model the 'treatment' variable (`design = ~ treatment`).  
各解析は1つの時点のサンプルだけを使うため、モデル化するのは 'treatment' 変数のみです（`design = ~ treatment`）。

### Notes
#### 注記

> **Interpreting "significant" genes**. Comparing the sets of significant hits at P0 vs P14 is suggestive but is **not** a valid test of whether the knockout's effect differs between timepoints. "Significant at P14 but not at P0" is not the same as "significantly different between P14 and P0." To actually test that, we will need the multifactorial model introduced in **2.1.3** .  
> **「有意」な遺伝子の解釈について**。P0 と P14 で有意なヒットの集合を比較するのは示唆的ですが、ノックアウトの効果が時点間で異なるかどうかの妥当な検定では **ありません**。「P14 では有意だが P0 では有意でない」ことは、「P14 と P0 の間で有意に異なる」ことと同じではありません。それを実際に検定するには、**2.1.3** で導入する多因子モデルが必要になります。

In [ ]:
# Helper: filter a DESeq results object to significant hits and report up/down counts.
get_hits <- function(deseq_result,decreasing=FALSE,outfile=NULL){
    res_df <- as.data.frame(deseq_result)  # Convert DESeqResults object to a data frame
    res_df <- res_df[order(res_df$log2FoldChange, decreasing=decreasing),]  # Sort by foldChange
    # Filter by padj < 0.05
    filtered_sorted_res <- res_df[(!is.na(res_df$padj)) & (res_df$padj < 0.05), ]

    # Count and report significant up- and down-regulated hits
    n_up <- sum(filtered_sorted_res$log2FoldChange > 0)
    n_down <- sum(filtered_sorted_res$log2FoldChange < 0)
    message(n_up, " upregulated and ", n_down, " downregulated hits (padj < 0.05)")

    if (!is.null(outfile)) {
        dir.create(dirname(outfile), recursive=TRUE, showWarnings=FALSE)
        write.table(res_df, file = outfile, sep = "\t", quote = FALSE)
    }
    return(filtered_sorted_res)
}

## 2.1.2.0 KO vs DKO at time P0
### 2.1.2.0 時点 P0 における KO 対 DKO

3 hits: 2 upregulated and 1 downregulated (padj < 0.05)  
3件のヒット：2件が発現上昇、1件が発現低下（padj < 0.05）

In [ ]:
# Subset to P0 samples
meta_p0 = meta %>% filter(timepoint == "P0")
data_p0 = data[, rownames(meta_p0)]

dds_p0 <- DESeq2::DESeqDataSetFromMatrix(countData = data_p0,
                              colData = meta_p0,
                              design = ~ treatment)
smallestGroupSize <- 5
keep <- rowSums(counts(dds_p0) >= 30) >= smallestGroupSize
dds_p0 <- dds_p0[keep,]
dds_p0 <- DESeq(dds_p0)
resultsNames(dds_p0)

In [ ]:
## Wald test with BH correction, then apeglm LFC shrinkage.
res_p0 <- results(dds_p0, name="treatment_DKO_vs_KO", independentFiltering=TRUE)
res_p0 <- lfcShrink(dds_p0, coef="treatment_DKO_vs_KO", res=res_p0, type="apeglm")

# get_hits(res_p0, outfile='results/atp8a2/P0_DKO_vs_KO_deg.tsv') %>% head(n=10)
get_hits(res_p0) %>% head(n=10)

In [ ]:
# PCA on P0 samples, colored by genotype.
vsd <- vst(dds_p0, blind=TRUE)

b = plotPCA(vsd, intgroup="treatment") +
    geom_label_repel(aes(label = name), size = 2.5, max.overlaps = Inf)
w=8;h=8
write_plot(b,outfile='results/atp8a2/pca_p0',width=w,height=h)
options(repr.plot.width=w, repr.plot.height=h)
b

## 2.1.2.1 KO vs DKO at P14
### 2.1.2.1 P14 における KO 対 DKO

130 hits: 59 upregulated and 71 downregulated (padj < 0.05)  
130件のヒット：59件が発現上昇、71件が発現低下（padj < 0.05）

In [ ]:
# Subset to P14 samples
meta_p14 = meta %>% filter(timepoint == "P14")
data_p14 = data[, rownames(meta_p14)]

dds_p14 <- DESeq2::DESeqDataSetFromMatrix(countData = data_p14,
                              colData = meta_p14,
                              design = ~ treatment)
smallestGroupSize <- 5
keep <- rowSums(counts(dds_p14) >= 30) >= smallestGroupSize
dds_p14 <- dds_p14[keep,]
dds_p14 <- DESeq(dds_p14)
resultsNames(dds_p14)

In [ ]:
## Wald test with BH correction, then apeglm LFC shrinkage.
res_p14 <- results(dds_p14, name="treatment_DKO_vs_KO", independentFiltering=TRUE)
res_p14 <- lfcShrink(dds_p14, coef="treatment_DKO_vs_KO", res=res_p14, type="apeglm")

# get_hits(res_p14, outfile='results/atp8a2/P14_DKO_vs_KO_deg.tsv') %>% head(n=10)
get_hits(res_p14) %>% head(n=10)

In [ ]:
# PCA on P14 samples, colored by genotype.
vsd <- vst(dds_p14, blind=TRUE)

c = plotPCA(vsd, intgroup="treatment") +
    geom_label_repel(aes(label = name), size = 2.5, max.overlaps = Inf)
w=8;h=8
write_plot(c,outfile='results/atp8a2/pca_p14',width=w,height=h)
options(repr.plot.width=w, repr.plot.height=h)
c

## 2.1.2.2 Comparing the two analyses: which genes are "the" hits?
### 2.1.2.2 2つの解析を比較する：「本当の」ヒットはどの遺伝子か？

We now have 2 models on different data asking similar questions: "which genes are differential in the DKO treatment?". These models use different data and different experimental conditions (P0 vs. P14), and therefore arrive at different sets of differential genes. This raises the question,  
いまや、異なるデータに対して似た問い「DKO 処理で発現変動する遺伝子はどれか？」を立てる2つのモデルがあります。これらのモデルは異なるデータと異なる実験条件（P0 対 P14）を使うため、異なる発現変動遺伝子の集合にたどり着きます。ここで次の問いが生じます。

> **Which set of differential genes is "best"?**   
> **どの発現変動遺伝子の集合が「最良」か？**

Furthermore, we might be interested in the related question,  
さらに、関連する次の問いにも関心があるかもしれません。

> **Which genes are specifically differential at P14 (or P0)?**  
> **P14（または P0）に特異的に発現変動する遺伝子はどれか？**

A very common way to handle a design like this is to run the comparison separately in each condition, as we have done above, and then overlap the two gene lists in a Venn diagram, shown below.  
このようなデザインを扱うごく一般的な方法は、上で行ったように各条件で別々に比較を実行し、その2つの遺伝子リストを、下に示すようなベン図で重ね合わせることです。

In [ ]:
library(ggVennDiagram)

# Significant genes (padj < 0.05) from a DESeq results object.
sig_genes <- function(res) rownames(as.data.frame(res))[which(!is.na(res$padj) & res$padj < 0.05)]

pairwise_sets <- list(
    "P0"  = sig_genes(res_p0),
    "P14" = sig_genes(res_p14)
)

# label_alpha = 1 gives the count labels a solid white background (the default 0.5
# lets the colored region show through, which is hard to read).
v2 <- ggVennDiagram(pairwise_sets, label = "count", label_alpha = 0) +
    scale_fill_gradient(low = "white", high = "firebrick") +
    theme(legend.position = "none")
w=6;h=5
write_plot(v2, outfile='results/atp8a2/venn_pairwise', width=w, height=h)
options(repr.plot.width=w, repr.plot.height=h)
v2

# The two ways people usually collapse this to a single gene list:
intersect_hits <- intersect(pairwise_sets$P0, pairwise_sets$P14)  # significant at BOTH timepoints
union_hits     <- union(pairwise_sets$P0, pairwise_sets$P14)      # significant at EITHER timepoint
cat("intersection:", length(intersect_hits), "genes\n")
cat("union:       ", length(union_hits), "genes\n")

Returning to the question, **which genes are differential in the DKO treatment?**, a couple of common solutions are:  
**DKO 処理で発現変動する遺伝子はどれか？** という問いに戻ると、一般的な解決策がいくつかあります。
- The **intersection** (2 genes, significant at *both* P0 and P14) is conservative, and  
  **積集合**（2遺伝子、P0 と P14 の *両方* で有意）は保守的であり、
- The **union** (131 genes, significant in *either*) is permissive, but...  
  **和集合**（131遺伝子、*いずれか* で有意）は寛容ですが…

Likewise, for the question, **which genes are specifically differential at P14?**, we might be tempted to use  
同様に、**P14 に特異的に発現変動する遺伝子はどれか？** という問いに対しては、次を使いたくなるかもしれません。
- The **difference** (128 genes, significant at P14 but not at P0).  
  **差集合**（128遺伝子、P14 では有意だが P0 では有意でない）。

However, "absence of evidence is not evidence of absence"<sup>1,2</sup>. In other words, we can't say that those 128 genes are not different at P0 just because *p* > 0.05.  
しかし、「証拠がないことは、存在しないことの証拠ではない」<sup>1,2</sup>。言い換えれば、単に *p* > 0.05 だからといって、それら128個の遺伝子が P0 で違いがないとは言えません。  
The multifactorial models in 2.1.3 give a better-founded alternative.  
2.1.3 の多因子モデルは、より根拠のしっかりした代替手段を与えてくれます。

**References**  
**参考文献**  
1: [Gelman & Stern, 2006](https://doi.org/10.1198/000313006X152649)  
2: [Altman & Bland, 1995](https://doi.org/10.1136/bmj.311.7003.485)

# 2.1.3 Multifactorial design
## 2.1.3 多因子デザイン

Instead of splitting the data and running two separate tests, we can model all 20 samples at once. This has two advantages: dispersion (per-gene variance) is estimated from all samples, and we can ask questions that span both timepoints.  
データを分割して2つの別々の検定を実行する代わりに、20サンプルすべてを一度にモデル化できます。これには2つの利点があります。分散（遺伝子ごとのばらつき）が全サンプルから推定されること、そして両方の時点にまたがる問いを立てられることです。

We will fit two models:  
2つのモデルを当てはめます。
- **Additive** (`~ timepoint + treatment`): estimates a single DKO-vs-KO effect, *pooled* across timepoints, while accounting for the (large) overall difference between P0 and P14.  
  **加法（Additive）**（`~ timepoint + treatment`）：P0 と P14 の間の（大きな）全体的な差を考慮しつつ、時点をまたいで *プールした* 単一の DKO 対 KO 効果を推定します。
- **Interaction** (`~ timepoint * treatment`): additionally lets the DKO-vs-KO effect *differ* between timepoints. A likelihood-ratio test of the interaction term asks, gene by gene, whether the knockout's effect depends on developmental stage, the formal version of the P0-vs-P14 question raised above.  
  **交互作用（Interaction）**（`~ timepoint * treatment`）：さらに、DKO 対 KO 効果が時点間で *異なる* ことを許します。交互作用項の尤度比検定は、遺伝子ごとに、ノックアウトの効果が発生段階に依存するかを問うもので、これは上で挙げた P0 対 P14 の問いを正式な形にしたものです。

### Notes
#### 注記

> **batch vs. biological effects.** Because all P0 samples were processed in batch 1, and all P14 in batch 2), a significant interaction cannot be cleanly attributed to *biology* vs *technical batch*. The genotype main effect (KO vs. DKO), estimated within timepoint, is not affected by this confounding. Best practice, if possible, would be to process replicates across different batches, so that batch effects can be corrected. However, this is not always possible, especially in a time series dataset like this one.  
> **バッチ効果と生物学的効果について。** すべての P0 サンプルはバッチ1で、すべての P14 サンプルはバッチ2で処理されたため、有意な交互作用を *生物学的なもの* と *技術的なバッチ* のどちらによるものか、はっきりと切り分けることはできません。時点内で推定される遺伝子型の主効果（KO 対 DKO）は、この交絡の影響を受けません。可能であれば、バッチ効果を補正できるよう、反復を異なるバッチにまたがって処理するのが最良の方法です。しかし、特にこのような時系列データセットでは、これが常に可能とは限りません。

> **Wald vs. LRT.** Notice the additive model (like the pairwise comparisons) uses the default **Wald** test, while the interaction model uses a **likelihood-ratio test (LRT)**. These suit two different kinds of question. The Wald test asks "is *this one* coefficient nonzero?", the natural fit when we want to *estimate and rank* a specific effect (the DKO-vs-KO log fold change), and it pairs with `lfcShrink`. The LRT instead compares a full model against a reduced one (here `~ timepoint * treatment` vs `~ timepoint + treatment`) and asks "do the extra term(s) improve the fit?", the natural fit for testing whether an *interaction belongs in the model at all*. For this 2×2 the interaction is a single coefficient, so a Wald test on it would give a nearly identical answer; we use the LRT because it is the idiomatic choice and generalizes to factors with more than two levels (e.g. a P0/P7/P14 time course), where it tests all interaction terms jointly in one shot.  
> **Wald 検定と LRT について。** 加法モデルは（ペアワイズ比較と同様に）既定の **Wald** 検定を使うのに対し、交互作用モデルは **尤度比検定（LRT）** を使うことに注目してください。これらは2つの異なる種類の問いに適しています。Wald 検定は「*この1つの* 係数はゼロでないか？」を問うもので、特定の効果（DKO 対 KO の log fold change）を *推定して順位づけ* したいときに自然で、`lfcShrink` と組み合わせて使えます。一方 LRT は、完全モデルを縮小モデル（ここでは `~ timepoint * treatment` 対 `~ timepoint + treatment`）と比較し、「追加した項はモデルの当てはまりを改善するか？」を問うもので、*そもそも交互作用をモデルに含めるべきか* を検定するのに自然です。この 2×2 では交互作用は単一の係数なので、それに対する Wald 検定はほぼ同じ答えを与えます。それでも LRT を使うのは、それが定石であり、2水準より多い因子（例：P0/P7/P14 の時系列）にも一般化でき、その場合すべての交互作用項を一度にまとめて検定できるからです。

## 2.1.3.0 Additive model: pooled DKO vs KO
### 2.1.3.0 加法モデル：プールした DKO 対 KO

28 hits: 26 upregulated and 2 downregulated (padj < 0.05)  
28件のヒット：26件が発現上昇、2件が発現低下（padj < 0.05）

In [ ]:
dds_add <- DESeq2::DESeqDataSetFromMatrix(countData = data,
                              colData = meta,
                              design = ~ timepoint + treatment)
smallestGroupSize <- 5
keep <- rowSums(counts(dds_add) >= 30) >= smallestGroupSize
dds_add <- dds_add[keep,]
dds_add <- DESeq(dds_add)
resultsNames(dds_add)

In [ ]:
## Pooled DKO-vs-KO effect, controlling for timepoint.
res_add <- results(dds_add, name="treatment_DKO_vs_KO", independentFiltering=TRUE)
res_add <- lfcShrink(dds_add, coef="treatment_DKO_vs_KO", res=res_add, type="apeglm")

# get_hits(res_add, outfile='results/atp8a2/additive_DKO_vs_KO_deg.tsv') %>% head(n=10)
get_hits(res_add) %>% head(n=10)

## 2.1.3.1 Comparing all three estimates of the DKO-vs-KO effect
### 2.1.3.1 DKO 対 KO 効果の3つの推定値を比較する

We now have three gene lists, each a different way of estimating "the DKO-vs-KO effect": at P0, at P14, and pooled across both (the additive model). Let's draw a three-way Venn to show how they relate:  
いまや3つの遺伝子リストがあり、それぞれ「DKO 対 KO 効果」を異なる方法で推定したものです。P0、P14、そして両方をプールしたもの（加法モデル）です。これらがどう関係するかを示すために、3要素のベン図を描きましょう。

In [ ]:
# Three estimates of the DKO-vs-KO effect: at P0, at P14, and pooled (additive).
three_sets <- list(
    "P0"       = sig_genes(res_p0),
    "P14"      = sig_genes(res_p14),
    "Additive" = sig_genes(res_add)
)

v3 <- ggVennDiagram(three_sets, label = "count", label_alpha = 0) +
    scale_fill_gradient(low = "white", high = "firebrick") +
    theme(legend.position = "none")
w=6;h=6
write_plot(v3, outfile='results/atp8a2/venn_three_way', width=w, height=h)
options(repr.plot.width=w, repr.plot.height=h)
v3

A few regions of interest:  
注目すべき領域がいくつかあります。
- **The additive model set** (28 genes): genes where combining all 20 samples gave enough statistical power to get a significant result. This is the set I would recommend reporting as "differential genes" w.r.t. the DKO treatment — and the interaction model below supports doing so, since it finds little evidence (only 5 genes) that the effect differs between timepoints, so a single pooled estimate is reasonable.  
  **加法モデルの集合**（28遺伝子）：20サンプルすべてを組み合わせたことで、有意な結果を得るのに十分な統計的検出力が得られた遺伝子です。これが、DKO 処理に関する「発現変動遺伝子」として報告するのを私が勧める集合です。下の交互作用モデルもこれを支持しており、効果が時点間で異なるという証拠はほとんど（わずか5遺伝子）見つからないため、単一のプールした推定値が妥当です。
- **Additive-only subset** (5 genes): genes the pooled analysis detects that neither pairwise analysis did. These are *mostly* attributable to the additional statistical power from using all of our data — though, as with any test, the set may also include some false positives.  
  **加法モデルのみの部分集合**（5遺伝子）：プールした解析では検出されたが、どちらのペアワイズ解析でも検出されなかった遺伝子です。これらは *ほとんど* が、全データを使うことで得られる追加の統計的検出力によるものです。ただし、どんな検定でもそうであるように、この集合にも偽陽性がいくらか含まれている可能性があります。
- **P14-only subset, outside the additive set** (108 genes): these could be false positives, P14-specific effects, or true positives that our additive model could not detect. From our data, it's difficult to know which.  
  **加法モデルの集合の外にある、P14 のみの部分集合**（108遺伝子）：これらは偽陽性、P14 に特異的な効果、あるいは加法モデルでは検出できなかった真陽性のいずれでもありえます。我々のデータからは、どれであるかを判断するのは困難です。
- **Core overlap subset** (2 genes): robust hits, including *Atp8a2* (the KO gene). That we recover the KO gene regardless of statistical treatment is a good confirmation that our experiment worked.  
  **中心の重なりの部分集合**（2遺伝子）：頑健なヒットで、*Atp8a2*（KO した遺伝子）を含みます。統計的な扱いに関わらず KO した遺伝子が検出されることは、実験がうまくいったことの良い確認になります。

## 2.1.3.2 Interaction model: does the effect depend on timepoint?
### 2.1.3.2 交互作用モデル：効果は時点に依存するか？

### Results
#### 結果
5 hits: 3 effects grew and 2 shrank (padj < 0.05)  
5件のヒット：3件は効果が大きくなり、2件は小さくなった（padj < 0.05）

In [ ]:
# Fit the full model with an interaction term and compare it, via a likelihood-ratio
# test (LRT), to the additive (reduced) model. The LRT p-value tests whether the
# interaction term improves the fit for each gene.
dds_int <- DESeq2::DESeqDataSetFromMatrix(countData = data,
                              colData = meta,
                              design = ~ timepoint * treatment)
smallestGroupSize <- 5
keep <- rowSums(counts(dds_int) >= 30) >= smallestGroupSize
dds_int <- dds_int[keep,]
dds_int <- DESeq(dds_int, test="LRT", reduced = ~ timepoint + treatment)
resultsNames(dds_int)

In [ ]:
## Genes with a significant genotype x timepoint interaction.
## Note: the LRT tests the interaction term; the reported log2FoldChange is the
## interaction coefficient (timepointP14:treatmentDKO), not shrunk here.
res_int <- results(dds_int, name="timepointP14.treatmentDKO", independentFiltering=TRUE)
get_hits(res_int) %>% head(n=10)

5 genes had significantly different effects at P14 than at P0. However, since all samples at P14 were done in a later batch, we might be concerned whether these are attributable to biology or technical "batch effect".  
5つの遺伝子では、効果が P0 よりも P14 で有意に異なっていました。しかし、P14 のサンプルはすべて後のバッチで処理されたため、これらが生物学的なものによるのか、それとも技術的な「バッチ効果」によるのか、懸念があるかもしれません。

## 2.1.4 Plotting DE genes in a volcano plot
### 2.1.4 ボルケーノプロットで発現変動遺伝子を可視化する

A differential expression result gives us two numbers per gene that we care about *together*: the **effect size** (the log2 fold change — how much, and in which direction, expression changed) and the **significance** (the adjusted p-value — how confident we are that the change is real). A gene with a huge fold change but a weak p-value is not convincing; neither is a gene with a tiny, if highly significant, change.  
発現変動の結果は、遺伝子ごとに *組み合わせて* 注目すべき2つの数値を与えてくれます。**効果量**（log2 fold change。発現がどれだけ、どの向きに変化したか）と、**有意性**（調整済み p 値。その変化が本物であるとどれだけ確信できるか）です。fold change が非常に大きくても p 値が弱い遺伝子は説得力がありません。逆に、非常に有意であっても変化がごくわずかな遺伝子も同様です。

A **volcano plot** shows both at once. Each point is a gene:  
**ボルケーノプロット** は、その両方を一度に示します。各点が1つの遺伝子です。
- the **x-axis** is the log2 fold change — points far from 0 changed a lot, points near 0 barely changed;  
  **x 軸** は log2 fold change です。0 から遠い点は大きく変化し、0 に近い点はほとんど変化していません。
- the **y-axis** is −log10(adjusted p-value) — points high up are highly significant, points near the bottom are not.  
  **y 軸** は −log10（調整済み p 値）です。上の方の点は非常に有意で、下の方の点はそうではありません。

The genes we usually care about — large *and* confident changes — sit in the upper corners, which gives the plot its erupting-"volcano" shape: up-regulated hits to the upper right, down-regulated hits to the upper left.  
通常注目する遺伝子、すなわち変化が大きく *かつ* 確信のもてる遺伝子は、上の両隅に位置します。これがこのプロットに噴火する「火山（volcano）」のような形を与えます。発現上昇のヒットは右上、発現低下のヒットは左上です。

We draw ours with **`EnhancedVolcano`**, a Bioconductor package that wraps `ggplot2` to produce customizable volcano plots. Because it returns a `ggplot` object, we can still customize it afterwards. Below we wrap it in a helper function `plot_volcano`, which takes a DESeq2 `results` table and formats a volcano plot. For this lesson, don't worry about the code, it's all formatting; the figure is the important part.  
ここでは **`EnhancedVolcano`** を使って描画します。これは `ggplot2` をラップして、カスタマイズ可能なボルケーノプロットを作る Bioconductor パッケージです。`ggplot` オブジェクトを返すので、後から自分でカスタマイズできます。以下では、DESeq2 の `results` テーブルを受け取ってボルケーノプロットを整形する補助関数 `plot_volcano` でラップしています。このレッスンでは、コードについては気にしないでください。すべて整形のためのものです。重要なのは図のほうです。

In [ ]:
library(EnhancedVolcano)

In [ ]:
ylabel = expression(-Log[10]*"("*italic(q)*")")

# Volcano plot that labels *every* significant hit (padj < 0.05), coloured by
# direction of change. Significance is by adjusted p-value alone (no fold-change
# cutoff), so we push EnhancedVolcano's FC cutoff lines off-screen.
#
# drop: optional character vector of gene names to omit from the plot. Useful for
# an outlier like Atp8a2 (the KO gene), whose padj is orders of magnitude smaller
# than every other hit and otherwise crushes the rest into the bottom of the plot.
# When genes are dropped, a note is added in the top-left corner naming them.
plot_volcano <- function(stats_df, drop = NULL){
    stats_df <- as.data.frame(stats_df)
    if (!is.null(drop)) stats_df <- stats_df[!rownames(stats_df) %in% drop, ]

    sig <- !is.na(stats_df$padj) & stats_df$padj < 0.05
    stats_df$highlight <- ifelse(sig, "sig", "other")
    # draw significant points last, so they sit on top
    stats_df <- stats_df[order(stats_df$highlight == "other", decreasing = TRUE), ]

    # red = significant & up, blue = significant & down, grey = not significant
    stats_df$color <- with(stats_df, ifelse(
        highlight == "sig" & log2FoldChange > 0, "red",
        ifelse(highlight == "sig" & log2FoldChange < 0, "dodgerblue", "grey50")))

    # label every significant gene
    sig_labels <- rownames(stats_df)[stats_df$highlight == "sig"]
    labSize = 6
    plt <- EnhancedVolcano(stats_df,
                lab = rownames(stats_df),
                selectLab = sig_labels,
                title = NULL,
                subtitle = NULL,
                caption = NULL,
                axisLabSize = 14,
                labSize = labSize,
                x = 'log2FoldChange',
                y = 'padj',
                pCutoff = 0.05,
                FCcutoff = 10,      # large + xlim below hides the (unused) FC cutoff lines
                xlim = c(-3,3),
                legendPosition = "none",
                pointSize = ifelse(stats_df$highlight == "sig", 3, 1),
                colCustom = setNames(stats_df$color, rownames(stats_df)),
                drawConnectors = TRUE,
                widthConnectors = 0.3,
                colConnectors = "grey60",
                maxoverlapsConnectors = Inf,
                colAlpha = ifelse(stats_df$highlight == "sig", .8, .6),
                ) %>% suppressWarnings

    p <- plt + ylab(ylabel)
    # Note in the top-left corner that the dropped gene(s) are off-scale.
    if (!is.null(drop)) {
        note <- paste0("(", paste(drop, collapse = ", "), " out of frame)")
        p <- p + annotate("text", x = -Inf, y = Inf, label = note,
                          hjust = -0.05, vjust = 1.3, size = labSize, fontface = "italic")
    }
    options(repr.plot.width=7, repr.plot.height=7)
    return(p)
}

In [ ]:
plt <- plot_volcano(res_add)
w=7;h=7
options(repr.plot.width=w, repr.plot.height=h)
write_plot(plt,"results/atp8a2/volcano",w,h)
plt
options(warn=0)

In [ ]:
options(warn=-1)
# Drop Atp8a2 (the KO gene): its padj is ~1e-71, orders of magnitude beyond every
# other hit, so including it crushes the rest into the bottom of the plot. It is the
# expected positive control and is already reported in the tables above.
plt <- plot_volcano(res_add, drop = "Atp8a2")
w=7;h=7
options(repr.plot.width=w, repr.plot.height=h)
write_plot(plt,"results/atp8a2/volcano_selected",w,h)
plt
options(warn=0)